# Phase 4 — Explainable Predictive Risk Scoring

**Goal:** predict a weekly risk score per station, with SHAP-based feature attribution
that's human-readable, not a black box.

**Validation approach:** this dataset has deliberately injected ground-truth patterns
(see `backend/generate_data.py`'s printed summary). Before trusting this model for
anything, we check that it actually recovers those known patterns. If it can't recover
patterns we know are there, it has no business being trusted on patterns we don't know
about.

**Honesty note up front:** this notebook reports what was actually found, including
one validation that came back weaker than hoped and had to be understood, not hidden.
That's the whole point of doing this validation step before wiring anything into the
app.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import shap
import json
import sys
sys.path.insert(0, '.')
from risk_explain import aggregate_shap_to_groups, generate_explanation_sentence

pd.set_option('display.max_columns', None)

## 1. Load station-week data

Built from raw incidents in `build_features.py`: for each (station, week), we have the
total incident count, a per-crime-type breakdown, and night/weekend share.

In [2]:
weekly = pd.read_pickle('weekly_station_data.pkl')
print(f"Shape: {weekly.shape}")
print(f"Weeks: {weekly['week'].min().date()} to {weekly['week'].max().date()}")
weekly.head()

Shape: (840, 14)
Weeks: 2024-01-01 to 2025-12-29


,station_id,week,total_count,Assault,Burglary,Chain Snatching,Cybercrime,Robbery,Theft,Vehicle Theft,night_share,weekend_share,station_name,district
0,1,2024-01-01,24,5,11,0,4,1,2,1,0.250000,0.458333,Cubbon Park PS,Bengaluru Urban
1,1,2024-01-08,23,6,0,0,3,2,5,7,0.173913,0.304348,Cubbon Park PS,Bengaluru Urban
2,1,2024-01-15,23,4,4,2,0,3,8,2,0.260870,0.304348,Cubbon Park PS,Bengaluru Urban
3,1,2024-01-22,34,3,3,4,2,5,14,3,0.147059,0.235294,Cubbon Park PS,Bengaluru Urban
4,1,2024-01-29,22,4,1,0,2,4,9,2,0.227273,0.545455,Cubbon Park PS,Bengaluru Urban


## 2. Feature engineering — with NO data leakage

This is a genuine **forecasting** setup: predicting week W's incident count using only
features computed from weeks strictly before W. Feeding the model week W's own
crime-type mix to predict week W's count would be leakage (you can't know a week's
composition before it happens) and would produce misleadingly good-looking metrics
that fall apart the moment this is used for anything real.

Feature groups:
- **Recent trend**: rolling 1/4/8-week incident counts, and a 4-vs-prior-4-week ratio
  (captures emerging spikes, conceptually similar to Phase 2's red-zone z-score)
- **Time-of-day pattern**: trailing night-incident share
- **Weekly pattern**: trailing weekend-incident share
- **Seasonal factor**: festival-season flag (Oct-Nov, matching the injected seasonal
  spike)
- **Crime-type mix**: trailing 8-week proportion of each crime type

In [3]:
CRIME_TYPES = ['Assault', 'Burglary', 'Chain Snatching', 'Cybercrime', 'Robbery', 'Theft', 'Vehicle Theft']

def add_lag_features(g):
    g = g.copy()
    g["lag1_total"] = g["total_count"].shift(1)
    g["roll4_total"] = g["total_count"].shift(1).rolling(4, min_periods=1).mean()
    g["roll8_total"] = g["total_count"].shift(1).rolling(8, min_periods=1).mean()
    g["roll4_prev4_ratio"] = g["roll4_total"] / g["total_count"].shift(5).rolling(4, min_periods=1).mean().replace(0, np.nan)
    g["lag_night_share"] = g["night_share"].shift(1).rolling(8, min_periods=1).mean()
    g["lag_weekend_share"] = g["weekend_share"].shift(1).rolling(8, min_periods=1).mean()
    for ct in CRIME_TYPES:
        col = f"mix_{ct}"
        g[col] = (g[ct].shift(1).rolling(8, min_periods=1).sum() /
                  g["total_count"].shift(1).rolling(8, min_periods=1).sum().replace(0, np.nan))
    return g

weekly = weekly.sort_values(["station_id", "week"]).reset_index(drop=True)
weekly = weekly.groupby("station_id", group_keys=False).apply(add_lag_features, include_groups=False).join(weekly[["station_id"]])
weekly["month"] = pd.to_datetime(weekly["week"]).dt.month
weekly["is_festival_season"] = weekly["month"].isin([10, 11]).astype(int)

FEATURE_COLS = ["lag1_total", "roll4_total", "roll8_total", "roll4_prev4_ratio",
                 "lag_night_share", "lag_weekend_share", "is_festival_season"] + [f"mix_{ct}" for ct in CRIME_TYPES]

model_df = weekly.dropna(subset=FEATURE_COLS + ["total_count"]).reset_index(drop=True)
print(f"Modeling rows (after dropping the warm-up period needed to compute lags): {len(model_df)} / {len(weekly)}")

Modeling rows (after dropping the warm-up period needed to compute lags): 800 / 840


## 3. Train/test split — time-based, not random

A random split would let the model train on weeks *after* the weeks it's tested on,
which is invalid for forecasting and would inflate test performance artificially.
We split chronologically: train on the earlier 75% of weeks, test on the most recent
25%.

In [4]:
split_week = model_df["week"].quantile(0.75, interpolation="nearest")
train = model_df[model_df["week"] < split_week]
test = model_df[model_df["week"] >= split_week]
print(f"Train: {len(train)} rows (< {pd.Timestamp(split_week).date()})")
print(f"Test:  {len(test)} rows (>= {pd.Timestamp(split_week).date()})")

X_train, y_train = train[FEATURE_COLS], train["total_count"]
X_test, y_test = test[FEATURE_COLS], test["total_count"]

model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

print(f"\nTrain MAE: {mean_absolute_error(y_train, pred_train):.2f}, R2: {r2_score(y_train, pred_train):.3f}")
print(f"Test  MAE: {mean_absolute_error(y_test, pred_test):.2f}, R2: {r2_score(y_test, pred_test):.3f}")

naive_pred = test["lag1_total"]
print(f"Naive baseline (predict last week's count) Test MAE: {mean_absolute_error(y_test, naive_pred):.2f}")

Train: 592 rows (< 2025-07-07)
Test:  208 rows (>= 2025-07-07)



Train MAE: 2.41, R2: 0.725
Test  MAE: 4.91, R2: 0.155
Naive baseline (predict last week's count) Test MAE: 6.66


**Reading these numbers honestly:** test R² of ~0.14 is modest — weekly crime
counts are inherently noisy (Poisson-distributed), and with 8 stations over ~2 years
there's a real ceiling on how predictable any individual week is. What matters more
than the raw R² is that **the model beats the naive baseline** (predicting last week's
count), and that it recovers the specific patterns we know are really there — which is
what we check next.

## 4. Validation — does the model recover our known injected patterns?

This dataset has 7 deliberately injected ground-truth patterns (base rates, weekend
skew, night skew, a seasonal spike, a static hotspot, a recent spike, an MO ring). We
check three of them here — the ones a station-week volume model should plausibly
detect.

### 4a. Feature importance — does the model actually rely on the features we'd expect?

In [5]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(importances.head(10))

is_festival_season     0.304880
mix_Vehicle Theft      0.181702
mix_Assault            0.060842
mix_Burglary           0.052706
mix_Theft              0.050669
mix_Robbery            0.048559
mix_Chain Snatching    0.047281
roll4_total            0.046158
roll8_total            0.038157
lag_weekend_share      0.038035
dtype: float32


**Result:** `is_festival_season` and `mix_Vehicle Theft` are the two most
important features by a wide margin — directly matching the injected Oct-Nov seasonal
spike and the Whitefield PS Vehicle Theft hotspot. This is a strong, clean signal.

### 4b. Whitefield PS (injected static Vehicle Theft hotspot) — is it predicted as highest-risk on average?

In [6]:
model_df["predicted"] = model.predict(model_df[FEATURE_COLS])
station_avg = model_df.groupby("station_name")["predicted"].mean().sort_values(ascending=False)
print(station_avg)

station_name
Whitefield PS         31.176523
Jayanagar PS          25.973833
Mangaluru North PS    25.624229
Mangaluru South PS    25.286440
Cubbon Park PS        25.002661
Krishnaraja PS        24.847689
Devaraja PS           24.843216
Yeshwanthpur PS       24.686207
Name: predicted, dtype: float32


**Result:** Whitefield PS is correctly predicted as the highest average-risk
station, standing clearly apart from the rest (~31 vs ~24-26 for others). This
directly validates recovery of the injected hotspot.

### 4c. Jayanagar PS (injected recent Robbery spike, last 4 weeks of the dataset) —
does the model pick it up?

This one came back weaker on first check, and that's worth being honest about rather
than glossing over.

In [7]:
jaya = model_df[model_df["station_name"] == "Jayanagar PS"].sort_values("week")
print(jaya[jaya["week"] >= "2025-12-01"][["week", "total_count", "predicted", "roll4_prev4_ratio", "mix_Robbery"]].to_string(index=False))

      week  total_count  predicted  roll4_prev4_ratio  mix_Robbery
2025-12-01           32  24.220829           1.139130     0.048780
2025-12-08           23  25.878309           1.101695     0.064516
2025-12-15           35  25.600044           0.918699     0.088983
2025-12-22           24  25.340822           0.866667     0.107143
2025-12-29            8  30.443165           0.870229     0.138776


**Honest finding:** total predicted *station volume* doesn't show a dramatic jump
here, because Robbery is only one of 7 crime types — a jump from ~2/week to ~6/week in
Robbery specifically is a real signal (this is exactly what Phase 2's red-zone z-score
alerting correctly flagged), but it's diluted when the model predicts *total* incidents
across all crime types (~25-30/week baseline). This is an expected, explicable
limitation of an aggregate-volume model, not a bug — see the SHAP check below for the
more precise picture.

### 4d. SHAP check — does the *explanation* pick up the spike even if the aggregate number doesn't move much?

In [8]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(model_df[FEATURE_COLS])
base_value = explainer.expected_value

target_idx = jaya[jaya["week"] == "2025-12-29"].index[0]
row_pos = model_df.index.get_loc(target_idx)
contribs = pd.Series(shap_values[row_pos], index=FEATURE_COLS).sort_values(key=abs, ascending=False)
print(f"Predicted: {model_df.loc[target_idx, 'predicted']:.1f}, base value: {base_value:.1f}\n")
print(contribs.head(6))

Predicted: 30.4, base value: 25.8

mix_Robbery           4.439607
lag1_total            1.010540
mix_Vehicle Theft    -0.511240
mix_Burglary         -0.351826
is_festival_season   -0.227436
lag_night_share       0.197643
dtype: float32


**Result:** by the final injected-spike week, `mix_Robbery` is by far the single
largest SHAP contributor (dwarfing every other feature) — the model's crime-type-mix
feature uses a trailing 8-week window, so it takes a few weeks of sustained pattern to
dominate that rolling average. That's sensible behavior for a feature meant to
represent a stable "mix," not a same-week alarm.

**Conclusion on validation 4c/4d together:** this risk model and Phase 2's red-zone
alerting are complementary, not redundant — z-score alerting is the right tool for an
immediate week-to-week spike; this model is the right tool for slower-building
structural risk (chronic hotspots, seasonal patterns). Worth saying exactly this in
the demo.

## 5. SHAP explanation, aggregated into human-readable groups

Raw per-feature SHAP values (17 of them) aren't demo-friendly. `risk_explain.py`
groups them into 5 categories and generates a plain-English sentence — this is the
piece that gets wired into the frontend.

In [9]:
# Example: Jayanagar's final spike week
grouped = aggregate_shap_to_groups(shap_values[row_pos], FEATURE_COLS)
for g in grouped:
    print(f"  {g['label']}: {g['pct']}% ({g['direction']})")
print()
print(generate_explanation_sentence("Jayanagar PS", model_df.loc[target_idx, "predicted"], base_value, grouped))

  crime-type mix: 67.1% (increasing)


  recent incident trend: 21.9% (increasing)
  seasonal factor: 4.2% (decreasing)
  time-of-day pattern: 3.6% (increasing)
  weekday/weekend pattern: 3.2% (decreasing)

Jayanagar PS shows elevated predicted risk (30 incidents/week vs a station-average baseline of 26), driven primarily by crime-type mix (67.1%) and recent incident trend (21.9%).


## 6. Save model artifacts for the API

These get loaded by `backend/main.py`'s `/api/risk-score` endpoint.

In [10]:
model.save_model("risk_model.json")
model_df.to_pickle("model_df.pkl")
with open("feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)
print("Saved risk_model.json, model_df.pkl, feature_cols.json")

Saved risk_model.json, model_df.pkl, feature_cols.json


## Summary

- Model beats a naive last-week-count baseline on held-out (chronologically later) data
- Feature importance and station-average validations cleanly recover the injected
  seasonal spike and static hotspot patterns
- The recent-spike pattern is recovered too, but only once enough weeks have passed
  for the rolling crime-type-mix feature to reflect it — an honest, explainable
  limitation rather than a hidden one
- SHAP explanations are grouped into 5 human-readable categories with a generated
  plain-English sentence, ready to wire into the API/frontend